In [ ]:

import cv2
import mediapipe as mp
import numpy as np
import time


# MediaPipe aliases
BaseOptions = mp.tasks.BaseOptions
HandLandMarker = mp.tasks.vision.HandLandmarker
HandLandMarkerOptions = mp.tasks.vision.HandLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode


# Load hand landmark model
model_path = "/home/manish1209-op/DataScience/OpenCV/Models/hand_landmarker.task"

model = HandLandMarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=RunningMode.VIDEO,
    num_hands=1
)

detector = HandLandMarker.create_from_options(model)


# Drawing variables
previous_point = None
canvas = None
prev_time = 0


# Toolbar colors (BGR format)
toolbar_colors = {
    "blue": (255, 0, 0),
    "green": (0, 255, 0),
    "red": (0, 0, 255),
    "yellow": (0, 255, 255),
    "pink": (120, 0, 255),
    "white": (255, 255, 255),
    "eraser": (0, 0, 0)
}

brush_size = 5
eraser = 80
current_tool = "white"

button_width = 100
button_height = 70
color_names = list(toolbar_colors.keys())


# Hand connections
ConnectionPoints = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (13, 17), (17, 18), (18, 19),
    (19, 20)
]



cap = cv2.VideoCapture(0)
start_time = time.monotonic()


while True:
    success, frame = cap.read()

    if not success:
        print("Unable to read webcam frame!")
        break

    frame = cv2.flip(frame, 1)
    frame = cv2.resize(frame, (960, 720))

    if canvas is None:
        canvas = np.zeros_like(frame)

    # Convert BGR to RGB for MediaPipe
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    # Video timestamp in milliseconds
    timestamp_ms = int(
        (time.monotonic() - start_time) * 1000
    )

    current_time = time.monotonic()

    fps = (
        1 / (current_time - prev_time)
        if prev_time != 0
        else 0
    )

    prev_time = current_time

    cv2.putText(
        frame,
        f"FPS: {fps:.1f}",
        (20, 110),
        cv2.FONT_HERSHEY_COMPLEX,
        0.7,
        (255, 0, 0),
        2
    )

    # Detect hands
    results = detector.detect_for_video(
        mp_image,
        timestamp_ms
    )

    for i, (name, color) in enumerate(toolbar_colors.items()):
        x1 = i * button_width
        x2 = x1 + button_width

        cv2.rectangle(
            frame,
            (x1, 0),
            (x2, button_height),
            color,
            -1
        )

    height, width = frame.shape[:2]

    # Process detected hand
    if results.hand_landmarks:

        hand = results.hand_landmarks[0]

        index_fingertip = hand[8]
        ix = int(index_fingertip.x * width)
        iy = int(index_fingertip.y * height)

        current_point = (ix, iy)

        thumb_tip = hand[4]
        tx = int(thumb_tip.x * width)
        ty = int(thumb_tip.y * height)

        # Calculate distance between thumb and index finger
        dx = ix - tx
        dy = iy - ty

        distance = np.hypot(dx, dy)

        pinched = distance < 40

        if pinched and iy < button_height:

            selected_index = ix // button_width

            if 0 <= selected_index < len(color_names):
                current_tool = color_names[selected_index]
                previous_point = None

        # Detect which fingers are raised
        index_up = hand[8].y < hand[6].y
        middle_up = hand[12].y < hand[10].y
        ring_up = hand[16].y < hand[14].y
        pinky_up = hand[20].y < hand[18].y

        index_only = (
            index_up
            and not middle_up
            and not ring_up
            and not pinky_up
        )

        all_up = (
            index_up
            and middle_up
            and ring_up
            and pinky_up
        )

        if current_tool == "eraser":
            thickness = eraser
        else:
            thickness = brush_size

        # Draw while only the index finger is raised
        if index_only:

            if previous_point is not None:
                cv2.line(
                    canvas,
                    previous_point,
                    current_point,
                    toolbar_colors[current_tool],
                    thickness
                )

            previous_point = current_point

        else:
            previous_point = None

    else:
        previous_point = None



    cv2.putText(
        frame,
        "Eraser"
        (630, 35),
        cv2.FONT,
        scale,
        (255, 255, 255),
        text_thickness
    )

    # Draw hand landmarks and connections
    for hands in results.hand_landmarks:

        landmarks_points = []

        for landmark in hands:
            X = int(landmark.x * width)
            Y = int(landmark.y * height)

            landmarks_points.append((X, Y))

            cv2.circle(
                frame,
                (X, Y),
                7,
                (0, 0, 255),
                -1
            )

        for start, end in ConnectionPoints:
            cv2.line(
                frame,
                landmarks_points[start],
                landmarks_points[end],
                (255, 255, 255),
                2
            )

    # Combine webcam frame and drawing canvas
    output = cv2.add(frame, canvas)

    cv2.imshow("Cam", output)

    # Press Q to quit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# Release resources
cap.release()
detector.close()
cv2.destroyAllWindows()
